# 8. KItem Contexts

## 8.1. Setting up

Before you run this tutorial: make sure to have access to a DSMS-instance of your interest, along with installation of this package, and have established access to the DSMS through DSMS-SDK (refer to [Connecting to DSMS](../dsms_sdk.md#connecting-to-dsms))

Import the needed classes and functions.

In [1]:
from dsms import DSMS, KItem

Now source the environmental variables from an `.env` file and start the DSMS-session.

In [2]:
import os
dsms = DSMS(env=".env") if os.path.exists(".env") else DSMS()

## 8.2. Putting KItems into contexts

KItems can not only be linked, but also put into a context in order to group and cluster them. This can especially become handy, when a set of KItems are related e.g. to a specific process, catalog, etc.

A KItem that is designated as a context (e.g. a `Project`) can group other KItems. A KItem joins a context by setting its `contexts` field to include the context KItem.

We will create a `Project` as the context and a `Dataset` as its member.

In [3]:
# Project is a context-capable KType (its spec has context=True)
project = KItem(
    name="Test project",
    ktype_id=dsms.ktypes.Project,
)

dsms.add(project)
dsms.commit()

print(project.url)

https://nash.materials-data.space/knowledge/project/testproject-e0654a2c


> **Note:** Some fields visible in KItem outputs (`authors`, `rdf_exists`, `user_groups`) are deprecated in v5.0.0 and are no longer populated by the server. They remain in the model for backward compatibility. Use `access_properties` for access control.

In [4]:
# Assign the dataset to the project context via the contexts field
dataset = KItem(
    name="Test dataset",
    ktype_id=dsms.ktypes.Dataset,
    contexts=[project],
)

dsms.add(dataset)
dsms.commit()

print(dataset.url)

https://nash.materials-data.space/knowledge/dataset/testdataset-340d5940


We can verify the context relationship from both directions: checking that the dataset reports its context, and checking that a freshly-fetched dataset shows `has_contexts=True`.

In [5]:
# From the dataset side: which contexts does this dataset belong to?
print("Dataset contexts:", dataset.contexts)

# Verify via a fresh fetch
refreshed_dataset = dsms[str(dataset.id)]
print("Dataset has_contexts:", refreshed_dataset.has_contexts)

Dataset contexts: [kitem:
  id: e0654a2c-68f7-4398-800a-bf56afd156b5
  name: Test project
  ktype_id: project
  slug: testproject-e0654a2c
  avatar_exists: false
  has_contexts: false
]
Dataset has_contexts: False


Now we can inspect the project:

In [6]:
project

kitem:
  id: e0654a2c-68f7-4398-800a-bf56afd156b5
  name: Test project
  ktype_id: project
  slug: testproject-e0654a2c
  avatar_exists: false
  has_contexts: false
  annotations: []
  attachments: []
  linked_kitems: []
  affiliations: []
  authors: []
  contacts: []
  created_at: 2026-06-07 20:53:30.905592
  updated_at: 2026-06-07 20:53:30.905592
  external_links: []
  apps: []
  rdf_exists: false
  access_properties:
    visibility: private
    user_access:
    - role: OWNER
      user_id: 6be66d9f-1a9f-44fc-8176-f71155de06ba
    group_access: []
  contexts: []

## 8.3. Searching within a context

`DSMS.search()` accepts a `contexts` parameter: a list of KItem IDs. The search returns only KItems that belong to at least one of the specified contexts.

In [7]:
# Search for KItems inside the project context
results = dsms.search(contexts=[str(project.id)])
for r in results:
    print(r.kitem.name, "| has_contexts:", r.kitem.has_contexts)

Test dataset | has_contexts: False


The `has_contexts` field on `KItemCompactedModel` is a boolean that indicates whether the KItem belongs to at least one context. It is populated by the server and available on search results without needing to fetch the full KItem.

## 8.4. Context-scoped SPARQL queries

Two SPARQL methods on `SparqlInterface` operate within the scope of a context KItem rather than the full triplestore.

`sparql_interface.query_context(context_id, query)` sends a standard SPARQL SELECT query scoped to the given context and returns a JSON result object (same format as `sparql_interface.query()`).

`sparql_interface.graph_context(context_id, query)` sends a graph query scoped to the context and returns a JSON graph result.

In [8]:
sparql_query = """
SELECT ?s ?p ?o
WHERE {
    ?s ?p ?o .
}
LIMIT 10
"""

try:
    results = dsms.sparql_interface.query_context(str(project.id), sparql_query)
    print(results)
except RuntimeError as e:
    print(f"Note: context SPARQL requires RDF knowledge graphs for context members.")
    print(f"Error: {e}")

Note: context SPARQL requires RDF knowledge graphs for context members.
Error: Context SPARQL query was not successful: {"detail":"No KG.kitem.ttl attachments found for context members. Ensure knowledge graphs have been generated for the members."}


## 8.5. Cleanup

Delete the KItems created during this tutorial.

In [9]:
del dsms[dataset]
del dsms[project]
dsms.commit()